# Excel / CSV to Adobe Illustrator JSON Data Converter
### Interactive Data Pre-Processing Pipeline for Universal Bulk Artwork Generator (`Illustrator_Bulk_Artwork_Generator.jsx`)

[![Python](https://img.shields.io/badge/Python-3.8%2B-blue.svg)](https://www.python.org/)
[![File Picker](https://img.shields.io/badge/UI-Native%20File%20Dialog-success.svg)]()
[![Engine](https://img.shields.io/badge/Pandas-Data%20Cleansing-150458.svg)](https://pandas.pydata.org/)
[![Target Script](https://img.shields.io/badge/Output-Illustrator%20JSX-FF9A00.svg)](file:///D:/New%20folder/PY/Illustrator_Bulk_Artwork_Generator.jsx)
[![Format](https://img.shields.io/badge/Supports-CSV%20%7C%20XLSX%20%7C%20XLS-green.svg)]()

---

## Executive Overview
This Jupyter Notebook automates the pre-processing and conversion of spreadsheet datasets (**Excel `.xlsx`/`.xls`** or **CSV `.csv`**) into the exact JSON format required by Adobe Illustrator's [`Illustrator_Bulk_Artwork_Generator.jsx`](file:///D:/New%20folder/PY/Illustrator_Bulk_Artwork_Generator.jsx).

### Key Features:
* **Interactive File Dialog:** Prompts you with a native Windows Explorer file picker to browse and select any `.xlsx`, `.xls`, or `.csv` file on your machine.
* **Smart Parent Folder Routing:** Automatically determines the parent directory and saves the exported JSON file directly into the parent folder with matching file naming.
* **Float Pollution Defense:** Fixes integer numbers (like ID `101`) that import as floats (`101.0`) when columns contain blank rows.
* **Ugly `NaN` Elimination:** Replaces missing/blank cells with empty strings (`""`) so Illustrator text frames clear cleanly without printing literal `"NaN"`.
* **Whitespace & Typography Normalization:** Strips trailing/leading spaces from names and text.
* **Serial Number Zero-Padding:** Formats IDs automatically (e.g., `"1"` -> `"001"`).
* **Live Blueprint & Simulation:** Displays a ready-to-use cheat sheet of your `{{placeholders}}` and simulates live text frame replacement.

---

## End-to-End Workflow Pipeline

```
┌─────────────────────────────────┐
│   Interactive User File Dialog  │
│   Pick Excel (.xlsx) or CSV     │
└────────────────┬────────────────┘
                 │
                 ▼
┌─────────────────────────────────┐
│   Data Cleansing & Sanitization │
│   * Strip leading/trailing space│
│   * Convert NaN/None -> ""      │
│   * Format floats 101.0 -> 101  │
│   * Optional ID padding 1 -> 001│
│   * Standardize Date strings    │
└────────────────┬────────────────┘
                 │
                 ▼
┌─────────────────────────────────┐
│   Schema & Placeholder Audit    │
│   * Verify JSON Array of Objects│
│   * Generate {{placeholder}} map│
│   * Live Template Substitution  │
└────────────────┬────────────────┘
                 │
                 ▼
┌─────────────────────────────────┐
│   JSON Export to Parent Folder  │
│   Saves UTF-8 '<name>.json'     │
└────────────────┬────────────────┘
                 │
                 ▼
┌─────────────────────────────────┐
│   Adobe Illustrator (JSX)       │
│   Clones 100s of Artboards!     │
└─────────────────────────────────┘
```

---

## Interactive Table of Contents
1. [Step 1: Install & Import Required Libraries](#-step-1-install--import-required-libraries)
2. [Step 2: Interactive File Selection & Parent Folder Setup](#-step-2-interactive-file-selection--parent-folder-setup)
3. [Step 3: Dataset Ingestion & Auto-Encoding Detection](#-step-3-dataset-ingestion--auto-encoding-detection)
4. [Step 4: Data Cleansing & Normalization Engine](#-step-4-data-cleansing--normalization-engine)
5. [Step 5: Placeholder Mapping & Compatibility Validator](#-step-5-placeholder-mapping--compatibility-validator)
6. [Step 6: JSON Export & Verification in Parent Folder](#-step-6-json-export--verification-in-parent-folder)
7. [Step 7: Bonus — Instant Sample Dataset Generator](#-step-7-bonus--instant-sample-dataset-generator)
8. [Step 8: Adobe Illustrator Execution Runbook](#-step-8-adobe-illustrator-execution-runbook)


---
## Step 1: Install & Import Required Libraries

This step imports the foundational libraries for data science, Excel/CSV ingestion, native file pickers, and JSON handling:
* `pandas`: High-performance data manipulation and cleaning.
* `openpyxl`: Ingestion engine for modern Excel (`.xlsx`) workbooks.
* `tkinter`: Standard Python GUI library used to display native Windows file browser dialogs.
* `json`: Standard JSON serialization conforming to RFC 8259.
* `IPython.display`: Interactive HTML tables and telemetry visualization.


In [8]:
# ============================================================
# STEP 1: LIBRARY IMPORTS & RUNTIME VERIFICATION
# ============================================================

import os
import sys
import json
import re
from pathlib import Path
from datetime import datetime
import pandas as pd
from IPython.display import display, HTML

# Verify Tkinter for native Windows File Dialog
try:
    import tkinter as tk
    from tkinter import filedialog
    TK_AVAILABLE = True
except ImportError:
    TK_AVAILABLE = False

print("=" * 65)
print("ENVIRONMENT DIAGNOSTICS & LIBRARY VERSIONS")
print("=" * 65)
print(f"Python Runtime   : {sys.version.split()[0]}")
print(f"Pandas Version   : {pd.__version__}")
try:
    import openpyxl
    print(f"Openpyxl Version : {openpyxl.__version__}")
except ImportError:
    print(" Openpyxl is not installed. Run: !pip install openpyxl")
print(f"Native GUI Dialog: {'AVAILABLE' if TK_AVAILABLE else 'UNAVAILABLE (Fallback to text input)'}")
print(f"Session Time     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 65)


ENVIRONMENT DIAGNOSTICS & LIBRARY VERSIONS
Python Runtime   : 3.13.14
Pandas Version   : 3.0.6
Openpyxl Version : 3.1.5
Native GUI Dialog: AVAILABLE
Session Time     : 2026-10-12 09:57:18


---
## Step 2: Interactive File Selection & Parent Folder Setup

When you run the code cell below:
1. **Interactive File Picker:** A native Windows Explorer File Dialog will open automatically. Navigate to your desired `.xlsx`, `.xls`, or `.csv` file and select it.
2. **Fallback Mode:** If the dialog is cancelled or running without a display, you can provide the path manually via `MANUAL_FILE_PATH` or pick from local files detected in the folder.
3. **Parent Folder Routing:** The script automatically computes the **parent directory** of the selected file, deriving the output path (e.g. `<parent_folder>/<filename>.json`).

### Configuration Reference

| Option | Type | Default | Description |
|:---|:---:|:---:|:---|
| `MANUAL_FILE_PATH` | `str` | `""` | Leave empty `""` to open file picker dialog, or enter path manually. |
| `SHEET_NAME` | `str/int` | `0` | Name or zero-based index of Excel worksheet (ignored for CSV). |
| `STANDARDIZE_KEYS` | `str` | `"trim"` | Options: `"trim"` (clean spaces), `"preserve"`, `"lower"`, `"snake_case"`. |
| `HANDLE_NAN_AS` | `str` | `""` | Value to substitute for blank/missing cells (empty string recommended for Illustrator). |
| `PAD_ID_COLUMNS` | `dict` | `{"id": 3, "serial": 4}` | Automatically left-pads specified numeric columns with zeros (e.g. `7` -> `"007"`). |
| `DATE_FORMAT` | `str` | `"%B %d, %Y"` | Date format string (e.g. `"%B %d, %Y"` produces `"October 12, 2026"`). |


In [9]:
# ============================================================
# STEP 2: INTERACTIVE FILE SELECTION & PARENT FOLDER SETUP
# ============================================================

# Optional: If you want to bypass the file dialog and specify a file directly,
# enter its path here (leave as "" to open the native Windows file picker dialog):
MANUAL_FILE_PATH = ""

def prompt_user_for_file():
    """
    Opens a native Windows File Open Dialog to prompt user for an Excel or CSV file.
    Falls back gracefully if cancelled or if GUI is unavailable.
    """
    if MANUAL_FILE_PATH and os.path.exists(MANUAL_FILE_PATH):
        print(f"Using pre-configured manual path: {MANUAL_FILE_PATH}")
        return os.path.abspath(MANUAL_FILE_PATH)

    selected_file = ""
    if TK_AVAILABLE:
        try:
            root = tk.Tk()
            root.withdraw()                    # Hide the background root window
            root.attributes('-topmost', True)   # Bring file picker dialog to foreground
            print("Opening native Windows File Dialog... Please select your Excel (.xlsx/.xls) or CSV file.")
            selected_file = filedialog.askopenfilename(
                title="Select Excel or CSV Spreadsheet",
                filetypes=[
                    ("Excel & CSV Spreadsheets", "*.xlsx;*.xls;*.csv"),
                    ("Excel Workbooks (*.xlsx, *.xls)", "*.xlsx;*.xls"),
                    ("CSV Documents (*.csv)", "*.csv"),
                    ("All Files (*.*)", "*.*")
                ]
            )
            root.destroy()
        except Exception as e:
            print(f"Dialog exception: {e}")

    # Fallback if user cancelled dialog or GUI failed
    if not selected_file or not os.path.exists(selected_file):
        print("No file selected from dialog. Checking local folder for spreadsheets...")
        candidates = [
            f for f in os.listdir(".")
            if f.lower().endswith(('.xlsx', '.xls', '.csv')) and not f.startswith(('~$', '.'))
        ]
        if candidates:
            selected_file = os.path.abspath(candidates[0])
            print(f"Automatically selected local spreadsheet: '{candidates[0]}'")
        else:
            default_sample = "sample_conference_badges.xlsx"
            selected_file = os.path.abspath(default_sample)
            print(f"Defaulting to sample dataset: '{default_sample}'")

    return selected_file

# Execute file selection
INPUT_FILE = prompt_user_for_file()

# Sheet name or index for Excel files (0 = first sheet)
SHEET_NAME = 0

# Column header normalization style ("trim", "preserve", "lower", "snake_case")
STANDARDIZE_KEYS = "trim"

# Replacement string for blank or missing (NaN/None) cells
HANDLE_NAN_AS = ""

# Zero-padding dictionary for serial/ID columns: { "column_name": length }
PAD_ID_COLUMNS = {"id": 3, "serial": 3, "reg_no": 4}

# Date format string for datetime columns
DATE_FORMAT = "%B %d, %Y"

# ============================================================
# SMART PARENT FOLDER ROUTING
# ============================================================
input_path_obj = Path(INPUT_FILE)
parent_folder = input_path_obj.parent
upper_parent_folder = parent_folder.parent
base_stem = input_path_obj.stem
json_filename = f"{base_stem}.json"

# Target Output Path inside the parent folder of the input file
OUTPUT_JSON_FILE = str(parent_folder / json_filename)
UPPER_PARENT_JSON_FILE = str(upper_parent_folder / json_filename)

print("=" * 65)
print("FILE SELECTION & ROUTING INITIALIZED:")
print("=" * 65)
print(f"Selected Input File : {INPUT_FILE}")
print(f"File Parent Folder  : {parent_folder}")
print(f"Destination JSON    : {OUTPUT_JSON_FILE}")
print(f"Header Style        : {STANDARDIZE_KEYS}")
print("=" * 65)


Opening native Windows File Dialog... Please select your Excel (.xlsx/.xls) or CSV file.
FILE SELECTION & ROUTING INITIALIZED:
Selected Input File : D:/duet-cse-fest-raffle/final_year.xlsx
File Parent Folder  : D:\duet-cse-fest-raffle
Destination JSON    : D:\duet-cse-fest-raffle\final_year.json
Header Style        : trim


---
## Step 3: Dataset Ingestion & Auto-Encoding Detection

This step reads the selected file into a Pandas DataFrame.
* For **CSV files**, it automatically tests common character encodings (`utf-8`, `utf-8-sig` with BOM, `cp1252`, and `latin1`) to prevent decoding crashes.
* For **Excel files**, it utilizes `openpyxl` to extract worksheet tables accurately.
* Displays a summary of total rows, column headers, and memory footprints.


In [10]:
# ============================================================
# STEP 3: DATASET INGESTION & AUTO-ENCODING RESOLUTION
# ============================================================

def load_dataset(file_path, sheet=0):
    """
    Intelligently loads CSV or Excel files with automated fallback handling.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Input file not found: {os.path.abspath(file_path)}")

    _, ext = os.path.splitext(file_path)
    ext = ext.lower()

    if ext in ['.xlsx', '.xls']:
        print(f"Loading Excel Workbook: '{os.path.basename(file_path)}' (Sheet: {sheet})")
        df = pd.read_excel(file_path, sheet_name=sheet)
    elif ext == '.csv':
        print(f"Loading CSV Document: '{os.path.basename(file_path)}'")
        encodings = ['utf-8-sig', 'utf-8', 'cp1252', 'latin1']
        df = None
        for enc in encodings:
            try:
                df = pd.read_csv(file_path, encoding=enc)
                print(f"   Successfully decoded using: {enc}")
                break
            except (UnicodeDecodeError, Exception):
                continue
        if df is None:
            raise ValueError(f"Could not decode CSV file '{file_path}' with standard encodings.")
    else:
        raise ValueError(f"Unsupported file format '{ext}'. Must be .csv, .xlsx, or .xls.")

    return df

try:
    df_raw = load_dataset(INPUT_FILE, SHEET_NAME)
    print(f"\nIngestion Successful:")
    print(f"   Total Records (Rows) : {len(df_raw):,}")
    print(f"   Total Attributes     : {len(df_raw.columns)}")
    print(f"   Column List          : {list(df_raw.columns)}")
    
    display(df_raw.head(5))
except FileNotFoundError:
    print(f"Notice: '{INPUT_FILE}' does not exist on disk.")
    print("Scroll down to Step 7 to generate a sample dataset, then re-run Step 2 and Step 3!")


Loading Excel Workbook: 'final_year.xlsx' (Sheet: 0)

Ingestion Successful:
   Total Records (Rows) : 121
   Total Attributes     : 2
   Column List          : ['name', 'student_id']


,name,student_id
0,Tanjina Akter,204017
1,Nayan Chandra,214001
2,Md. Palash,214004
3,Md. Rion Hossen,214005
4,Boktiar Mia,214006


---
## Step 4: Data Cleansing & Normalization Engine

This step applies sanitization to eliminate formatting bugs before passing data to Illustrator:
1. **Header Trimming & Normalization:** Cleans column keys to match Illustrator placeholders seamlessly.
2. **Whitespace Stripping:** Removes accidental leading/trailing spaces from all string values.
3. **Float Cleanup:** Replaces float artifacts like `101.0` with clean integer strings `101`.
4. **Missing Value Sanitization:** Replaces `NaN`, `None`, and `<NA>` with `""` so Illustrator doesn't print literal `"NaN"` on badges.
5. **Serial Zero-Padding:** Applies configured padding rules (e.g. `PAD_ID_COLUMNS`).
6. **Date Formatting:** Converts datetime objects into clean, readable date strings.


In [11]:
# ============================================================
# STEP 4: DATA CLEANSING & PRE-PROCESSING ENGINE
# ============================================================

def clean_column_name(col, style="trim"):
    """Standardizes column header names according to selected style."""
    col_str = str(col).strip()
    if style == "lower":
        return col_str.lower()
    elif style == "snake_case":
        cleaned = re.sub(r'\s+', '_', col_str)
        cleaned = re.sub(r'[^a-zA-Z0-9_]', '', cleaned)
        return cleaned.lower()
    elif style == "trim":
        return col_str
    return col_str

def clean_record_value(val, col_name, pad_rules=None, date_fmt="%B %d, %Y", nan_val=""):
    """Sanitizes individual cell values into clean string or number representations."""
    if pd.isna(val) or val is None:
        return nan_val

    if isinstance(val, (pd.Timestamp, datetime)):
        return val.strftime(date_fmt)

    if isinstance(val, float) and val.is_integer():
        val = int(val)

    if pad_rules and isinstance(pad_rules, dict):
        col_lower = str(col_name).strip().lower()
        for rule_col, length in pad_rules.items():
            if col_lower == str(rule_col).strip().lower():
                try:
                    num_val = int(float(val))
                    return f"{num_val:0{length}d}"
                except (ValueError, TypeError):
                    pass

    if isinstance(val, str):
        return val.strip()

    return val

df_clean = df_raw.copy()
df_clean.columns = [clean_column_name(c, STANDARDIZE_KEYS) for c in df_clean.columns]

cleaned_records = []
for _, row in df_clean.iterrows():
    record = {}
    for col in df_clean.columns:
        val = row[col]
        cleaned_val = clean_record_value(
            val,
            col,
            pad_rules=PAD_ID_COLUMNS,
            date_fmt=DATE_FORMAT,
            nan_val=HANDLE_NAN_AS
        )
        record[col] = cleaned_val
    cleaned_records.append(record)

df_sanitized = pd.DataFrame(cleaned_records)

print("Data Cleansing Completed:")
print(f"   Cleaned Records Count: {len(cleaned_records):,}")
display(df_sanitized.head(5))


Data Cleansing Completed:
   Cleaned Records Count: 121


,name,student_id
0,Tanjina Akter,204017
1,Nayan Chandra,214001
2,Md. Palash,214004
3,Md. Rion Hossen,214005
4,Boktiar Mia,214006


---
## Step 5: Placeholder Mapping & Compatibility Validator

This step validates that your dataset satisfies **every single requirement** enforced by [`Illustrator_Bulk_Artwork_Generator.jsx`](file:///D:/New%20folder/PY/Illustrator_Bulk_Artwork_Generator.jsx):
* **Root Entity:** Confirms root is a non-empty `list` (`Array`).
* **Record Structure:** Confirms every record is a valid `dict` (`Object`).
* **Placeholder Blueprint:** Outputs a ready-to-use cheat sheet of the exact `{{placeholder}}` strings to copy into your Adobe Illustrator text frames.
* **Simulated Text Frame Replacement:** Previews how a sample template string will resolve with real data.


In [12]:
# ============================================================
# STEP 5: SCHEMA VALIDATOR & TEMPLATE BLUEPRINT
# ============================================================

assert isinstance(cleaned_records, list), "ERROR: Root must be a list (Array in JSX)!"
assert len(cleaned_records) > 0, "ERROR: Dataset contains 0 records!"
for i, r in enumerate(cleaned_records):
    assert isinstance(r, dict), f"ERROR: Record {i+1} must be an object/dict!"

print("=" * 65)
print("COMPATIBILITY VERIFICATION: 100% SATISFIED FOR Illustrator_Bulk_Artwork_Generator.jsx")
print("=" * 65)

columns = list(cleaned_records[0].keys())
blueprint_data = []
first_record = cleaned_records[0]

for col in columns:
    placeholder = f"{{{{{col}}}}}"
    sample_val = str(first_record.get(col, ""))
    blueprint_data.append({
        "Spreadsheet Column": col,
        "Illustrator Text Frame Tag": placeholder,
        "Sample Output (Row 1)": sample_val
    })

df_blueprint = pd.DataFrame(blueprint_data)
print("\nCOPY-PASTE BLUEPRINT FOR ADOBE ILLUSTRATOR:")
display(df_blueprint)

print("─" * 65)
print("LIVE ILLUSTRATOR TEMPLATE SUBSTITUTION PREVIEW:")
sample_template = " // ".join([f"{col}: {{{{{col}}}}}" for col in columns[:4]])
print(f"Master Template Frame : {sample_template}")

simulated_result = sample_template
for col in columns:
    tag = f"{{{{{col}}}}}"
    simulated_result = simulated_result.replace(tag, str(first_record.get(col, "")))

print(f"Rendered Output (Item 1): {simulated_result}")
print("─" * 65)


COMPATIBILITY VERIFICATION: 100% SATISFIED FOR Illustrator_Bulk_Artwork_Generator.jsx

COPY-PASTE BLUEPRINT FOR ADOBE ILLUSTRATOR:


,Spreadsheet Column,Illustrator Text Frame Tag,Sample Output (Row 1)
0,name,{{name}},Tanjina Akter
1,student_id,{{student_id}},204017


─────────────────────────────────────────────────────────────────
LIVE ILLUSTRATOR TEMPLATE SUBSTITUTION PREVIEW:
Master Template Frame : name: {{name}} // student_id: {{student_id}}
Rendered Output (Item 1): name: Tanjina Akter // student_id: 204017
─────────────────────────────────────────────────────────────────


---
## Step 6: JSON Export & Verification in Parent Folder

This step exports your sanitized records to a formatted JSON file on disk inside the **parent folder**:
* **Parent Folder Destination:** Saved directly to `OUTPUT_JSON_FILE` in the parent directory of your input file.
* **Secondary Mirror Copy:** Also saves a synchronized copy into the workspace parent directory (`UPPER_PARENT_JSON_FILE`) so you can select the file from either location.
* **UTF-8 Encoding:** Guarantees universal compatibility with international characters, symbols, and diacritics.


In [13]:
# ============================================================
# STEP 6: JSON EXPORT & INTEGRITY AUDITING IN PARENT FOLDER
# ============================================================

# 1. Save in the Parent Folder of the Input File
os.makedirs(os.path.dirname(OUTPUT_JSON_FILE), exist_ok=True)
with open(OUTPUT_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(cleaned_records, f, indent=2, ensure_ascii=False)

file_bytes = os.path.getsize(OUTPUT_JSON_FILE)
size_kb = file_bytes / 1024

print("=" * 65)
print("JSON EXPORT SUCCESSFUL (SAVED IN PARENT FOLDER)!")
print("=" * 65)
print(f"File Location       : {OUTPUT_JSON_FILE}")
print(f"Parent Folder       : {os.path.dirname(OUTPUT_JSON_FILE)}")
print(f"Total Records       : {len(cleaned_records):,} records")
print(f"File Size           : {size_kb:.2f} KB ({file_bytes:,} bytes)")

# 2. Also save synchronized copy in workspace upper parent directory if distinct
if os.path.abspath(OUTPUT_JSON_FILE) != os.path.abspath(UPPER_PARENT_JSON_FILE):
    try:
        os.makedirs(os.path.dirname(UPPER_PARENT_JSON_FILE), exist_ok=True)
        with open(UPPER_PARENT_JSON_FILE, "w", encoding="utf-8") as f_upper:
            json.dump(cleaned_records, f_upper, indent=2, ensure_ascii=False)
        print(f"Upper Parent Mirror : {UPPER_PARENT_JSON_FILE}")
    except Exception as e:
        pass

print("=" * 65)
print("You can now run Adobe Illustrator -> File -> Scripts -> Other Script...")
print(f"   Select 'Illustrator_Bulk_Artwork_Generator.jsx', then choose:")
print(f"   '{OUTPUT_JSON_FILE}'!")


JSON EXPORT SUCCESSFUL (SAVED IN PARENT FOLDER)!
File Location       : D:\duet-cse-fest-raffle\final_year.json
Parent Folder       : D:\duet-cse-fest-raffle
Total Records       : 121 records
File Size           : 8.26 KB (8,459 bytes)
Upper Parent Mirror : D:\final_year.json
You can now run Adobe Illustrator -> File -> Scripts -> Other Script...
   Select 'Illustrator_Bulk_Artwork_Generator.jsx', then choose:
   'D:\duet-cse-fest-raffle\final_year.json'!


---
## Step 7: Bonus — Instant Sample Dataset Generator

Need sample data to test the workflow immediately? Run this cell to generate two ready-to-use sample spreadsheets:
1. **`sample_conference_badges.xlsx`**: Conference attendee dataset with `id`, `name`, `organization`, `designation`, `category`, and `table`.
2. **`sample_certificates.csv`**: Student course completion certificates with `serial`, `student_name`, `course_title`, `grade`, and `date`.


In [14]:
# ============================================================
# STEP 7: GENERATE SAMPLE SPREADSHEETS (FOR IMMEDIATE TESTING)
# ============================================================

badges_data = [
    {"id": 1, "name": "Dr. Ada Lovelace", "organization": "Babbage Analytical", "designation": "Keynote Speaker", "category": "VIP", "table": "A-1"},
    {"id": 2, "name": "Alan Turing", "organization": "Bletchley Informatics", "designation": "AI Researcher", "category": "Speaker", "table": "A-2"},
    {"id": 3, "name": "Grace Hopper", "organization": "Naval Systems & Compilers", "designation": "Distinguished Fellow", "category": "VIP", "table": "A-3"},
    {"id": 4, "name": "Claude Shannon", "organization": "Bell Telecommunications", "designation": "Information Theorist", "category": "Delegate", "table": "B-1"},
    {"id": 5, "name": "Katherine Johnson", "organization": "Orbital Trajectory Lab", "designation": "Lead Mathematician", "category": "VIP", "table": "A-4"},
    {"id": 6, "name": "John von Neumann", "organization": "Advanced Architecture Inst.", "designation": "Systems Architect", "category": "Speaker", "table": "B-2"},
    {"id": 7, "name": "Margaret Hamilton", "organization": "Apollo Software Engineering", "designation": "Director of Systems", "category": "Keynote", "table": "A-5"},
    {"id": 8, "name": "Donald Knuth", "organization": "TeX & Algorithmics Foundation", "designation": "Professor Emeritus", "category": "Delegate", "table": "B-3"}
]

df_badges = pd.DataFrame(badges_data)
excel_sample_path = "sample_conference_badges.xlsx"
df_badges.to_excel(excel_sample_path, index=False)
print(f"Generated sample Excel workbook: '{os.path.abspath(excel_sample_path)}'")

certs_data = [
    {"serial": 101, "student_name": "Eleanor Vance", "course_title": "Advanced Microprocessors & 8085 Interfacing", "grade": "High Distinction", "date": "October 2026"},
    {"serial": 102, "student_name": "Marcus Holloway", "course_title": "Network Security & Cryptography", "grade": "Distinction", "date": "October 2026"},
    {"serial": 103, "student_name": "Clara Oswald", "course_title": "Quantum Computing & Quantum Information", "grade": "High Distinction", "date": "October 2026"},
    {"serial": 104, "student_name": "Arthur Dent", "course_title": "Space Systems Engineering & Navigation", "grade": "Credit", "date": "October 2026"}
]

df_certs = pd.DataFrame(certs_data)
csv_sample_path = "sample_certificates.csv"
df_certs.to_csv(csv_sample_path, index=False, encoding="utf-8-sig")
print(f"Generated sample CSV dataset   : '{os.path.abspath(csv_sample_path)}'")

print("\nTip: Run Step 2 again to select this file via the file dialog!")


Generated sample Excel workbook: 'd:\New folder\PY\sample_conference_badges.xlsx'
Generated sample CSV dataset   : 'd:\New folder\PY\sample_certificates.csv'

Tip: Run Step 2 again to select this file via the file dialog!


---
## Step 8: Adobe Illustrator Execution Runbook

Once you have generated your JSON file in the parent folder, follow these simple steps to generate your bulk artwork:

### 1. Template Preparation (`.ai` file)
* Open Adobe Illustrator and design your master layout on **Artboard 1**.
* In your text frames, insert tags corresponding to your JSON keys:
  * e.g., `{{name}}`, `{{id}}`, `{{organization}}`, `{{table}}`
* Format typography (fonts, alignment, colors, font sizes) exactly as you want the final output to appear.
* Save the template file as an Illustrator document (e.g. `Master_Badge_Template.ai`).

### 2. Running `Illustrator_Bulk_Artwork_Generator.jsx`
1. Open Adobe Illustrator.
2. Go to **File > Scripts > Other Script...** (or press `Ctrl+F12` on Windows / `Cmd+F12` on macOS).
3. Select [`D:\New folder\PY\Illustrator_Bulk_Artwork_Generator.jsx`](file:///D:/New%20folder/PY/Illustrator_Bulk_Artwork_Generator.jsx).
4. When prompted:
   * **Select JSON file:** Pick the exported JSON file saved in your parent folder.
   * **Select Illustrator template:** Pick your `Master_Badge_Template.ai`.
   * **Enter output file name:** Confirm or enter a base name (e.g., `Badges_Print_Batch`).
5. Watch Illustrator automatically clone artboards, duplicate layers, center artwork, and replace all text dynamically!

---
*Generated by the Universal Artwork Automation Suite.*
